In [1]:
pip install tensorflow opencv-python matplotlib

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\VERRA HALIZZAH\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [1]:
import os
import shutil
import random
import hashlib
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
DATASET_RAW = "DatasetBatikRaw"
DATASET_SPLIT = "DatasetBatikFix"
WORK_DIR = "Dataset_Audit"


TRAIN_RATIO = 0.7
VAL_RATIO = 0.2
TEST_RATIO = 0.1
RANDOM_SEED = 42

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-8
os.makedirs(WORK_DIR, exist_ok = True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Config Siap")


Config Siap


In [6]:
valid_ext = {".jpg", ".jpeg", ".png"}
records = []
corrupted_files = []

class_names = sorted([
    d for d in os.listdir(DATASET_RAW)
    if os.path.isdir(os.path.join(DATASET_RAW, d))
])

for class_name in class_names:
    class_dir = os.path.join(DATASET_RAW, class_name)
    files = sorted(os.listdir(class_dir))

    for fname in files:
        fpath = os.path.join(class_dir, fname)
        ext = Path(fname).suffix.lower()
        
        if ext not in valid_ext:
            continue
        
        try:
            with Image.open(fpath) as img:
                img.verify()
            
            with Image.open(fpath) as img:
                width, height = img.size
                mode = img.mode
                
            file_size = os.path.getsize(fpath)
            
            records.append({
                "class_name": class_name,
                "filename": fname,
                "filepath": fpath,
                "ext": ext,
                "width": width,
                "height": height,
                "aspect_ratio": round(width / height, 4) if height > 0 else None,
                "file_size_bytes": file_size,
                "mode": mode,
                "is_corrupt": False
            })
        except (UnidentifiedImageError, OSError, Exception) as e:
            corrupted_files.append({
                "class_name": class_name,
                "filename": fname,
                "filepath": fpath,
                "error": str(e)
            })

df = pd.DataFrame(records)
df_corrupt = pd.DataFrame(corrupted_files)

df.to_csv(os.path.join(WORK_DIR, "dataset_metadata.csv"), index=False)
df_corrupt.to_csv(os.path.join(WORK_DIR, "corrupted_files.csv"), index=False)

print(f"Total valid images: {len(df)}")
print(f"Total corrupted images: {len(df_corrupt)}")
print("Saved: dataset_metadata.csv, corrupted_files.csv") 

Total valid images: 3000
Total corrupted images: 0
Saved: dataset_metadata.csv, corrupted_files.csv


In [14]:
def file_md5(path, chunk_size=8192):
    md5 = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            md5.update(chunk)
    return md5.hexdigest()

hashes = []
for fp in df["filepath"]:
    hashes.append(file_md5(fp))

df["md5"] = hashes

dup_groups = df.groupby("md5").filter(lambda x: len(x) > 1).sort_values("md5")
dup_groups.to_csv(os.path.join(WORK_DIR, "duplicate_exact_files.csv"), index=False)

print(f"Total exact-duplicate entries: {len(dup_groups)}")
print("Saved: duplicate_exact_files.csv")
dup_groups.head(20)

Total exact-duplicate entries: 0
Saved: duplicate_exact_files.csv


,class_name,filename,filepath,ext,width,height,aspect_ratio,file_size_bytes,mode,is_corrupt,md5


In [15]:
cross_class_dup = (
    df.groupby("md5")["class_name"]
      .nunique()
      .reset_index(name="num_classes")
)

cross_class_dup = cross_class_dup[cross_class_dup["num_classes"] > 1]

if len(cross_class_dup) > 0:
    cross_class_detail = df[df["md5"].isin(cross_class_dup["md5"])]
    cross_class_detail.to_csv(os.path.join(WORK_DIR, "cross_class_duplicates.csv"), index=False)
    print(f"WARNING: ditemukan {len(cross_class_dup)} hash duplicate lintas kelas!")
    display(cross_class_detail.head(20))
else:
    print("Tidak ada exact duplicate lintas kelas.")

Tidak ada exact duplicate lintas kelas.


In [20]:
if os.path.exists(DATASET_SPLIT):
    shutil.rmtree(DATASET_SPLIT)
    
for split in ["train", "validation", "test"]:
    os.makedirs(os.path.join(DATASET_SPLIT, split), exist_ok = True)
    
split_records = []
    
for class_name in class_names:
    class_df = df[df["class_name"] == class_name].copy()
    
    # shuffle reproducible
    class_df = class_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    
    n_total = len(class_df)
    n_train = int(n_total * TRAIN_RATIO)
    n_val = int(n_total * VAL_RATIO)
    n_test = n_total - n_train - n_val
    
    train_df = class_df.iloc[:n_train]
    val_df = class_df.iloc[n_train:n_train+n_val]
    test_df = class_df.iloc[n_train+n_val:]
    
    split_map = [
        ("train", train_df),
        ("validation", val_df),
        ("test", test_df)
    ]
    
    for split_name, split_df in split_map:
        class_split_dir = os.path.join(
            DATASET_SPLIT, split_name, class_name)
        os.makedirs(class_split_dir, exist_ok=True)
        
        for _, row in split_df.iterrows():
            src = row["filepath"]
            dst = os.path.join(class_split_dir, row["filename"])
            shutil.copy2(src, dst)
            
            split_records.append({
                "class_name": class_name,
                "filename": row["filename"],
                "filepath": src,
                "split": split_name
            })

split_df_all = pd.DataFrame(split_records)
split_df_all.to_csv(os.path.join(WORK_DIR, "split_manifest.csv"), index=False)

print("Split reproducible selesai.")
print(split_df_all["split"].value_counts())


Split reproducible selesai.
split
train         2100
validation     600
test           300
Name: count, dtype: int64


In [21]:
summary = split_df_all.groupby(["split", "class_name"]).size().unstack(fill_value=0)
display(summary)

print("Train total:", (split_df_all["split"] == "train").sum())
print("Validation total:", (split_df_all["split"] == "validation").sum())
print("Test total:", (split_df_all["split"] == "test").sum())

class_name,batik_betawi,batik_bokor_kencono,batik_buketan,batik_dayak,batik_jlamprang,batik_kawung,batik_liong,batik_mega_mendung,batik_parang,batik_sekarjagad,batik_sidoluhur,batik_sidomukti,batik_sidomulyo,batik_singa_barong,batik_srikaton,batik_tribusono,batik_tujuh_rupa,batik_tuntrum,batik_wahyu_tumurun,batik_wirasat
split,,,,,,,,,,,,,,,,,,,,
test,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15,15
train,105,105,105,105,105,105,105,105,105,105,105,105,105,105,105,105,105,105,105,105
validation,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30,30


Train total: 2100
Validation total: 600
Test total: 300
